# Firing Rate Statistics Using Poisson Regression

This notebook contains code for reading in the organized spike data, running a series of Poisson regression models, and exporting the results.

---
> Justin Campbell & Dylan Jensen  
> justin.campbell@hsc.utah.edu  
> 10/23/2023

## 1. Import Libraries

In [1]:
import os
import glob
import pandas as pd
import statsmodels.api as sm

## 2. Load & Organize Data

In [10]:
stimDF = pd.read_csv('/Users/justincampbell/Library/CloudStorage/GoogleDrive-u0815766@gcloud.utah.edu/My Drive/Research Projects/BLAESUnits/BLAESUnits Stim.csv')
stimDF['StimLat'] = [stimDF.loc[i,'Cathode'][0] for i in range(len(stimDF))]

In [11]:
# TODO: Figure out why some rows are being dropped following merge

# Find PSCounts.csv files
PSCountFilepaths = glob.glob('/Volumes/Hippocampus/BLAESUnits/Results/*/PSCounts.csv')

# Concatenate files
PSCountFiles = []
for file in PSCountFilepaths:
    PSCountFiles.append(pd.read_csv(file, index_col = 0))
PSCountDF = pd.concat(PSCountFiles)
PSCountDF.reset_index(inplace = True, drop = True)

# Merge with stimDF
PSCountDF = PSCountDF.merge(stimDF[['pID','StimLat']], on = 'pID')

# # Add additional columns
PSCountDF['MicroID'] = [PSCountDF.loc[i,'Channel'] + '-' + str(PSCountDF.loc[i,'Unit']) for i in range(len(PSCountDF))] 
PSCountDF['UnitLat'] = [PSCountDF.loc[i,'Channel'][1] for i in range(len(PSCountDF))]
PSCountDF['Region'] = [PSCountDF.loc[i,'Channel'][2:-1] for i in range(len(PSCountDF))]

# # Clean up Region labels
print('Original Labels: ', PSCountDF['Region'].unique())
regionDict = {'CV': 'CV', 'HIP': 'HIP', 'ACC': 'ACC', 'CD': 'CD', 'OFC': 'OFC',
            'VCG': 'CV', 'AHIP': 'HIP', 'HC': 'HIP', 'HCA': 'HIP', 'DAC': 'ACC', 'DCG': 'CD'}
PSCountDF['Region'] = PSCountDF['Region'].map(regionDict)
print('Remapped Labels: ', PSCountDF['Region'].unique())
PSCountDF = PSCountDF.dropna(subset = ['Region'])
print('Final Labels: ', PSCountDF['Region'].unique())
PSCountDF

Original Labels:  ['CV' 'OFC' 'HIP' 'an10' 'an11' 'HC' 'AHIP' 'VCG' 'DAC' 'HCA' 'DCG' 'ACC'
 'CD']
Remapped Labels:  ['CV' 'OFC' 'HIP' nan 'ACC' 'CD']
Final Labels:  ['CV' 'OFC' 'HIP' 'ACC' 'CD']


,Pre,During,Post,pID,Channel,Unit,StimLat,MicroID,UnitLat,Region
0,8,6,9,UIC20220202,mLCV1,1,L,mLCV1-1,L,CV
1,40,56,63,UIC20220202,mLCV3,1,L,mLCV3-1,L,CV
2,8,4,3,UIC20220202,mLCV4,1,L,mLCV4-1,L,CV
3,5,4,3,UIC20220202,mLCV6,1,L,mLCV6-1,L,CV
4,76,87,123,UIC20220202,mLCV7,1,L,mLCV7-1,L,CV
...,...,...,...,...,...,...,...,...,...,...
100,3,1,2,UIC20221501,mROFC3,1,R,mROFC3-1,R,OFC
101,1,0,1,UIC20221501,mROFC6,1,R,mROFC6-1,R,OFC
102,168,144,115,UIC20221501,mRVCG4,1,R,mRVCG4-1,R,CV
103,22,25,14,UIC20221501,mRVCG4,2,R,mRVCG4-2,R,CV


In [4]:
# Melt the dataframe to convert from wide to long format
PSCountDFLong = PSCountDF.melt(id_vars = ['MicroID', 'pID', 'Region', 'UnitLat', 'StimLat'], value_vars = ['Pre', 'Post', 'During'], var_name = 'Condition', value_name = 'Spikes')

# Dummy code Condition variable
PSCountDFLong = pd.get_dummies(PSCountDFLong, columns = ['Condition'])

In [5]:
# df = pd.read_csv('/Users/justincampbell/Library/CloudStorage/GoogleDrive-u0815766@gcloud.utah.edu/My Drive/Research Projects/BLAESUnits/SpikeData.csv')
# # Melt the dataframe to convert from wide to long format
# df = df.melt(id_vars = ['Micro Id', 'Patient', 'Region', 'Lat', 'StimLat'], value_vars = ['Pre', 'Post', 'During'], var_name = 'Condition', value_name = 'Spikes')

# # Dummy code Condition variable
# df = pd.get_dummies(df, columns = ['Condition'])
# df

## 3. Run Poisson Regression

### 3.1 Test for Pre-/Post difference across all regions

In [6]:
X = PSCountDFLong[['Condition_During', 'Condition_Post']]
X = sm.add_constant(X)
y = PSCountDFLong['Spikes']

model = sm.GLM(y, X, family = sm.families.Poisson()).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 Spikes   No. Observations:                  276
Model:                            GLM   Df Residuals:                      273
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -76676.
Date:                Mon, 23 Oct 2023   Deviance:                   1.5173e+05
Time:                        13:48:12   Pearson chi2:                 4.26e+05
No. Iterations:                     6   Pseudo R-squ. (CS):           0.004417
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                5.4647      0.007  

### 3.2 Test for Pre-/Post difference in each region separately

In [7]:
regions = PSCountDFLong['Region'].unique().tolist()

for i in range(len(regions)):
    print('Region: %s' %regions[i])
    df_region = PSCountDFLong[PSCountDFLong['Region'] == regions[i]]
    
    X = df_region[['Condition_During', 'Condition_Post']]
    X = sm.add_constant(X)
    y = df_region['Spikes']

    model = sm.GLM(y, X, family = sm.families.Poisson()).fit()
    print(model.summary())

Region: CV
                 Generalized Linear Model Regression Results                  
Dep. Variable:                 Spikes   No. Observations:                   72
Model:                            GLM   Df Residuals:                       69
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -9399.3
Date:                Mon, 23 Oct 2023   Deviance:                       18394.
Time:                        13:48:12   Pearson chi2:                 2.38e+04
No. Iterations:                     5   Pseudo R-squ. (CS):            0.09653
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                5.0231  

### 3. Test for Pre-Post difference in *ipsilateral* vs *contralateral* regions

In [8]:
df_ipsi = PSCountDFLong[PSCountDFLong['UnitLat'] == PSCountDFLong['StimLat']]

X = df_ipsi[['Condition_During', 'Condition_Post']]
X = sm.add_constant(X)
y = df_ipsi['Spikes']

model = sm.GLM(y, X, family = sm.families.Poisson()).fit()
print(model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:                 Spikes   No. Observations:                  207
Model:                            GLM   Df Residuals:                      204
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -24510.
Date:                Mon, 23 Oct 2023   Deviance:                       47901.
Time:                        13:48:12   Pearson chi2:                 7.45e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1451
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                4.8520      0.011  

In [9]:
df_cont = PSCountDFLong[PSCountDFLong['UnitLat'] != PSCountDFLong['StimLat']]

X = df_cont[['Condition_During', 'Condition_Post']]
X = sm.add_constant(X)
y = df_cont['Spikes']

model = sm.GLM(y, X, family = sm.families.Poisson()).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 Spikes   No. Observations:                   69
Model:                            GLM   Df Residuals:                       66
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -34907.
Date:                Mon, 23 Oct 2023   Deviance:                       69311.
Time:                        13:48:12   Pearson chi2:                 1.45e+05
No. Iterations:                     5   Pseudo R-squ. (CS):             0.4200
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                6.3293      0.009  

## 4. Export